# Concurrency

Welcome! Concurrency lets your program do multiple things at once. Rust makes concurrency safer by design: the type system and ownership rules prevent common bugs like data races. This guide covers threads, message passing, shared state, mutexes, atomics, and the `Send` and `Sync` traits. By the end, you'll be ready to write concurrent Rust code with confidence.

## Threads: `std::thread::spawn`

A **thread** is a separate execution path. Use `std::thread::spawn` to create a new thread. The closure you pass runs in that thread. Use `join()` to wait for the thread to finish and get its result.

In [ ]:
use std::thread;
use std::time::Duration;

let handle = thread::spawn(|| {
    println!("Hello from the spawned thread!");
    thread::sleep(Duration::from_millis(100));
    42
});

println!("Hello from the main thread!");
let result = handle.join().unwrap();
println!("Spawned thread returned: {}", result);

## Move Closures

Closures passed to `spawn` must own their data. Use `move` to transfer ownership of variables into the closure. Without `move`, the closure would borrow — but the thread might outlive the main thread's stack, causing a dangling reference.

In [ ]:
use std::thread;

let v = vec![1, 2, 3];
let handle = thread::spawn(move || {
    println!("Here's a vector: {:?}", v);
});

handle.join().unwrap();
// v is now moved; we can't use it here

## Message Passing: mpsc Channels

Rust's standard approach: "Do not communicate by sharing memory; instead, share memory by communicating." The `mpsc` module (multiple producer, single consumer) provides channels. One thread sends; another receives.

In [ ]:
use std::sync::mpsc;
use std::thread;

let (tx, rx) = mpsc::channel();

thread::spawn(move || {
    tx.send("Hello from the thread!").unwrap();
});

let received = rx.recv().unwrap();
println!("Got: {}", received);

## Multiple Producers

You can clone the `Sender` to have multiple threads send to the same channel. Use `Sender::clone(&tx)` to create additional senders.

In [ ]:
use std::sync::mpsc;
use std::thread;

let (tx, rx) = mpsc::channel();
let tx2 = tx.clone();

thread::spawn(move || { tx.send(1).unwrap(); });
thread::spawn(move || { tx2.send(2).unwrap(); });

println!("Received: {}", rx.recv().unwrap());
println!("Received: {}", rx.recv().unwrap());

## Shared State: Mutex<T>

A **mutex** (mutual exclusion) ensures only one thread can access the data at a time. To access the data, you acquire the lock; when done, you release it. The lock is released automatically when the `MutexGuard` goes out of scope.

In [ ]:
use std::sync::Mutex;

let m = Mutex::new(0);
{
    let mut guard = m.lock().unwrap();
    *guard += 1;
}
println!("Value: {}", *m.lock().unwrap());

## Arc<Mutex<T>>: Shared Mutable State Across Threads

`Mutex` alone can't be shared across threads — you need to wrap it in `Arc` for shared ownership. `Arc<Mutex<T>>` is the classic pattern for "multiple threads, one mutable value."

In [ ]:
use std::sync::{Arc, Mutex};
use std::thread;

let counter = Arc::new(Mutex::new(0));
let mut handles = vec![];

for _ in 0..4 {
    let c = Arc::clone(&counter);
    let h = thread::spawn(move || {
        let mut num = c.lock().unwrap();
        *num += 1;
    });
    handles.push(h);
}

for h in handles {
    h.join().unwrap();
}
println!("Result: {}", *counter.lock().unwrap());

## RwLock<T>: Multiple Readers or One Writer

`RwLock` (Read-Write Lock) allows multiple readers *or* one writer. Use it when reads are common and writes are rare — it can be more efficient than `Mutex` in that case.

In [ ]:
use std::sync::RwLock;

let lock = RwLock::new(5);
{
    let r1 = lock.read().unwrap();
    let r2 = lock.read().unwrap();
    println!("Readers see: {} and {}", *r1, *r2);
}
{
    let mut w = lock.write().unwrap();
    *w += 1;
}
println!("After write: {}", *lock.read().unwrap());

## Send and Sync Traits

**Send:** Types that can be transferred across thread boundaries. Most types are `Send`; exceptions include `Rc` (use `Arc` for threads).

**Sync:** Types whose references can be shared across threads (`&T` is `Send` when `T: Sync`). `Mutex<T>` is `Sync` when `T: Send`.

The compiler enforces these automatically. If you try to use a non-`Send` type in a thread, you'll get a compile error.

## Atomic Types

**Atomics** are lock-free primitives for simple shared state. `AtomicBool`, `AtomicUsize`, `AtomicIsize`, etc. Use `Ordering` to specify memory ordering (e.g., `Ordering::SeqCst` for sequential consistency).

In [ ]:
use std::sync::atomic::{AtomicUsize, Ordering};

let counter = AtomicUsize::new(0);
counter.fetch_add(1, Ordering::SeqCst);
counter.fetch_add(1, Ordering::SeqCst);
println!("Counter: {}", counter.load(Ordering::SeqCst));

## Common Concurrency Patterns

1. **Pipeline:** Thread A sends to B, B sends to C — use multiple channels.
2. **Worker pool:** A channel of tasks; worker threads receive and process.
3. **Shared counter:** `Arc<Mutex<usize>>` or `Arc<AtomicUsize>`.
4. **Broadcast:** Multiple receivers (use crates like `crossbeam` or `tokio` for fan-out).

In [ ]:
// Worker pattern: channel + threads
use std::sync::mpsc;
use std::thread;

let (tx, rx) = mpsc::channel();
tx.send(10).unwrap();
tx.send(20).unwrap();
drop(tx);  // Signal no more sends

let handle = thread::spawn(move || {
    for msg in rx {
        println!("Worker received: {}", msg);
    }
});
handle.join().unwrap();

## Deadlock Prevention

**Deadlock** occurs when two or more threads wait for each other forever. To reduce risk:

- **Lock in a consistent order** — if thread A locks X then Y, thread B should also lock X then Y, never Y then X.
- **Avoid holding locks while doing I/O** — release the lock before blocking.
- **Use timeouts** — `try_lock()` or `lock_timeout()` instead of blocking forever.
- **Prefer channels** — message passing often avoids lock ordering issues.

## Rayon: Parallel Iterators (Brief Overview)

**Rayon** is a popular crate for data parallelism. It turns regular iterators into parallel ones with minimal code change. Add `rayon` to `Cargo.toml` and use `.par_iter()` instead of `.iter()`.

In [ ]:
// Rayon example (requires: rayon = "1.x" in Cargo.toml)
// Uncomment and run in a project with rayon:
// use rayon::prelude::*;
// let v: Vec<i32> = (0..1000).collect();
// let sum: i32 = v.par_iter().map(|x| x * 2).sum();
// println!("Parallel sum: {}", sum);

// Without Rayon, sequential version:
let v: Vec<i32> = (0..10).collect();
let sum: i32 = v.iter().map(|x| x * 2).sum();
println!("Sequential sum (concept): {}", sum);

## Common Mistakes Beginners Make

1. **Forgetting `move`** in thread closures — leads to borrow checker errors.
2. **Using `Rc` instead of `Arc`** for shared data across threads — `Rc` is not `Send`.
3. **Holding a lock too long** — can cause contention or deadlock.
4. **Ignoring `Ordering` in atomics** — `SeqCst` is safe but may be slower; understand the tradeoffs.
5. **Blocking in async code** — if using async runtimes, avoid blocking the executor.

## Key Rules to Remember

- Use `thread::spawn` for threads; `move` closures to transfer ownership.
- Prefer **message passing** (channels) over shared state when possible.
- For shared mutable state: `Arc<Mutex<T>>` or `Arc<RwLock<T>>`.
- `Send` = safe to transfer across threads; `Sync` = safe to share via `&T`.
- Use atomics for simple counters/flags; use `Ordering::SeqCst` when unsure.
- Lock in a consistent order to avoid deadlocks; prefer channels when you can.